# **walter**

## **Project Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import subprocess
import getpass

IN_COLAB = "google.colab" in sys.modules

REPO_NAME = "walter"
GIT_BRANCH = "main"
REPO_PATH = Path("/content") / REPO_NAME


def get_tokens():
    github_token = os.getenv("GITHUB_TOKEN") or getpass.getpass("GitHub token: ")
    return github_token


def install_core_ml_stack():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "transformers==4.44.2",
            "accelerate==0.33.0",
            "pyarrow",
        ],
        check=True,
    )


def setup_repo(github_token):
    os.chdir("/content")

    repo_url = f"https://{github_token}@github.com/Mango-Cats/{REPO_NAME}.git"

    if REPO_PATH.exists():
        os.chdir(REPO_PATH)
        subprocess.run(["git", "fetch", "origin"], check=True)
        subprocess.run(["git", "reset", "--hard", f"origin/{GIT_BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", repo_url], check=True)
        os.chdir(REPO_PATH)

    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)


if IN_COLAB:
    print("Colab detected")

    github_token = get_tokens()

    install_core_ml_stack()
    setup_repo(github_token)

    os.chdir(REPO_PATH)
    print("Project root:", REPO_PATH)

RES_DIR = "results/"

In [2]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)

Torch: 2.11.0+cpu
Device: cpu


## **Nomenclature and Terminologies**

The dataset $\mathcal{D}_{\text{raw}}$ (represented as `D_raw` in the source code) refers to the raw Philippine human-drug registry, which is freely available as a `.csv` file at [https://verification.fda.gov.ph/drug_productslist.php](https://verification.fda.gov.ph/drug_productslist.php).

The intermediate dataset, $\mathcal{D}_{\text{clean}}$ (`D_clean`), is the result of passing $\mathcal{D}_{\text{raw}}
$ through the preprocessing pipeline. 

The final datasets, $\mathcal{D}_{\text{train}}$ (`D_train`) and $\mathcal{D}_{\text{test}}$ (`D_test`), are used to train and test a weighted sum of similarity measures via a genetic algorithm. These consist of three columns: an ordered pair of drugs formed from the cleaned registry, such that every pair $(x, y) \in \mathcal{D}_{\text{clean}} \times \mathcal{D}_{\text{clean}}$, followed by their label: `p` for positive (LASA) and `u` for noise (unlabeled). These two datasets are derived from an 80-20 split of the union of two disjoint subsets:

*   **$P \subset (\mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{test}})$** (`P`) is the set of known positives, consisting of ordered drug pairs manually verified as LASA.
*   **$U \subset (\mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{test}})$** (`U`) is the unlabeled noise set, consisting of randomly paired drugs from $\mathcal{D}_{\text{clean}}$. $U$ acts as the noise class (where true labels are unknown), meaning it may contain undetected LASA pairs.

Furthermore, it is established that $|U| \gg |P|$, $P \cap U = \emptyset$, $P \cup U = \mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{test}}$, and $\mathcal{D}_{\text{test}} \cap \mathcal{D}_{\text{train}} = \emptyset$.

## **Preprocessing**

The first step is to preprocess (load, validate, clean) the FDA human drug registry dataset to construct our $\mathcal{D}_\text{clean}$ dataset.

Ensure that the FDA human drug registry dataset exists anywhere starting from the root folder and has the same filename defined by `PRIMARY_FNAME`.

The code for this section is located at [`/src/preprocessing.py`](/src/preprocessing.py).

The function `master_maker` is the coordinator function that performs data loading, validation, cleaning, and reporting. 

In [3]:
import pandas as pd
import src.preprocessing as pre

skip = True

if skip:
    filename = Path(pre.CLEANED_FNAME + ".parquet")
    D_clean = pd.read_parquet(filename)
else:
    D_clean = pre.master_maker(sort=True, save=True)

Let's look at the info of the dataset.

In [4]:
display(D_clean.head())
display(D_clean.shape)

,Brand Name
0,0.9% NaCl-Sapher
1,0.9% Sodchlorsaph
2,1 Ceeplus
3,1000Vc
4,2-Gen


(22838, 1)

Finally, let's look at a slice of 10 entries in the dataset by using the `get_rand_entries()` function.

In [5]:
display(pre.get_rand_entries(df=D_clean, count=10))

,Brand Name
20438,Tri-Senza
20439,Tri-V
20440,Tria-G
20441,Triacebact
20442,Triacef
20443,TRIAFORE
20444,Triagen
20445,Triagin
20446,Trialoc
20447,Triamax


## **True LASA Pairs**

Now that we have the $\mathcal{D}_\text{clean}$ we can now proceed with constructing $\mathcal{D}_\text{train}$. We will prioritize constructing the subset of true LASA pairs, or the set $P$.

The code for this section is located at [`/src/proposer/`](/src/proposer/).

### **Local Models**

In [6]:
from pandas import DataFrame, read_csv
from src.proposer.inference import load_inference, LocalModel, run_inference

USE_ISMP = True

if not USE_ISMP:
    ITERATION_COUNT = 400
    OUTPUT = RES_DIR + "lasa_run.json"

    result = run_inference(
        output_path=OUTPUT,
        D_clean=D_clean,
        model_choice=LocalModel.QWEN3_1_7B,
        iterations=ITERATION_COUNT,
    )
    P: DataFrame = load_inference(OUTPUT)
else:
    P: DataFrame = read_csv("data/ismp.csv")

In [7]:
display(P.head())
display(P.shape)

,Brand Name,Confusible
0,abelcet,amphotericin b
1,accupril,aciphex
2,acetaminophen,acetazolamide
3,acetazolamide,acetohexamide
4,acetic acid for irrigation,glacial acetic acid


(536, 2)

## **Noise Pairs**

Now that we have $P$, we can now complete constructing $\mathcal{D}_\text{train}$ by constructing the set $U$ or the unlabeled noise set.

The code for this section is located at [`/src/noise.py`](/src/noise.py).

In [8]:
import src.noise as noise

U = noise.make_noise(true_df=P, full_registry=D_clean)

<walter> P-vocabulary size:      784
<walter> Known positive pairs:   534
<walter> FDA registry size:      22,838
<walter> Target |U|:             16,020  (ratio 1:30)
<walter> Similarity threshold:   20 (ANY measure)
<walter> Tier 1 target:          10,413
<walter> Tier 2 target:          5,607

<walter> Building anchor-based hard negatives...
<walter> Candidates generated: 14,195,656

<walter> Building broader coverage (pre-sample size: 2,000)...
<walter> Candidates generated: 1,525,241

<walter> Final Tier 1 pairs:    10,413
<walter> Final Tier 2 pairs:    5,607
<walter> Total U:               16,020
<walter> Actual ratio (U/P):    1:30.0


In [9]:
display(U.head())
display(U.shape)

,Brand Name,Confusible,similarity,tier,Confusible_label
0,t-pa,Nelstac,51.428571,1,0
1,pristiq,Ambiclin,26.666667,1,0
2,syeda,NICOZOLE,30.000000,1,0
3,kineret,Bicindoxil,23.529412,1,0
4,razadyne,Plasmabloc,22.222222,1,0


(16020, 5)

## **Assembling**

Now that both subsets are complete. Assembling $\mathcal{D}_\text{train}$ is simply a concatenation of $P$ and $U$. 

In [10]:
from src.dataset import prepare_and_save_datasets

D = prepare_and_save_datasets(P, U, RES_DIR)


<walter> Combining P and U...
<walter> Combined size before cleaning: 16,556

<walter> Cleaning and deduplicating dataset...
<walter> Removed 11 self-pairs
<walter> Combined size after cleaning: 16,545
<walter> Removed auxiliary columns: ['similarity', 'tier']

<walter> Successfully saved dataset:
	- CSV: results/walter.csv
	- Parquet: results/walter.parquet

<walter> Dataset size: 16,545


In [11]:
display(D.head())
display(D.shape)

,Brand Name,Confusible,label
14083,lexyl od,agycin 500,0
3746,sinemet,broloxin,0
4655,dexmethylphenidate,flamilium,0
16405,metromin plus,astenzyd,0
15935,bioxime,budmed,0


(16545, 3)